# Capstone Topic 4: Orchestration & CI/CD

> **Phase 7 → Week 16 (Capstone) → Topic 4**

Wire up the complete ShopStream pipeline with an Airflow DAG and a GitHub Actions CI/CD workflow that tests, lints, and deploys the pipeline code to S3.

### Interview Cheat Sheet

**Q: Walk me through the Airflow DAG for ShopStream.**
> The DAG runs daily at 6am UTC with `catchup=False` and `max_active_runs=1`. It starts with an S3KeySensor waiting for Bronze files (15-min poke interval, reschedule mode to avoid holding a worker slot). Then a TaskGroup spins up an EMR cluster and runs 4 EMR Steps sequentially: validate_bronze → silver_transform → gold_aggregate → freshness_check. After all steps succeed, the cluster terminates with `trigger_rule='all_done'` (runs even on failure to avoid orphaned clusters). A final DQ task reads the JSON metrics written by the Spark job and raises if they indicate failure.

**Q: How do you make the pipeline idempotent so re-running for the same date is safe?**
> Three mechanisms: (1) Silver write uses `mode=overwrite` with `spark.sql.sources.partitionOverwriteMode=dynamic` — only the date partition is replaced. (2) Gold uses the same pattern. (3) Dead-letter uses `mode=append` because we want to accumulate all rejections over retries. The EMR cluster is fresh each run — no shared state. If the same DQ JSON is overwritten, it just overwrites with the same content. Net result: running the same DAG execution twice produces identical Silver and Gold tables.

## Part 1: Complete Production Airflow DAG

In [ ]:
print("""
# shopstream_pipeline.py  (Airflow DAG — deploy to s3://shopstream-code/dags/)
# ══════════════════════════════════════════════════════════════════════════════
from __future__ import annotations
import json
from datetime import datetime, timedelta

from airflow import DAG
from airflow.models import Variable
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.providers.amazon.aws.operators.emr import (
    EmrCreateJobFlowOperator, EmrAddStepsOperator, EmrTerminateJobFlowOperator,
)
from airflow.providers.amazon.aws.sensors.emr import EmrStepSensor
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
from airflow.utils.task_group import TaskGroup
from airflow.utils.trigger_rule import TriggerRule

# ── Constants ─────────────────────────────────────────────────────────────
ENV          = Variable.get("ENV", default_var="dev")
CODE_BUCKET  = Variable.get("CODE_BUCKET",  default_var="shopstream-code")
DATA_BUCKET  = Variable.get("DATA_BUCKET",  default_var="shopstream-data")
GIT_SHA      = Variable.get("GIT_SHA",      default_var="latest")
SUBNET_ID    = Variable.get("SUBNET_ID",    default_var="subnet-abc123")
LOG_URI      = f"s3://{DATA_BUCKET}/emr-logs/"
CODE_URI     = f"s3://{CODE_BUCKET}/{GIT_SHA}/libs.zip"

# ── EMR cluster config ────────────────────────────────────────────────────
EMR_CONFIG = {
    "Name":            "shopstream-{{ ds_nodash }}",
    "ReleaseLabel":    "emr-6.15.0",
    "LogUri":          LOG_URI,
    "Applications":    [{"Name": "Spark"}, {"Name": "Hadoop"}],
    "Instances": {
        "InstanceGroups": [
            {"Name": "Master", "InstanceRole": "MASTER",
             "InstanceType": "m5.xlarge", "InstanceCount": 1},
            {"Name": "Core",   "InstanceRole": "CORE",
             "InstanceType": "m5.2xlarge", "InstanceCount": 2},
            {"Name": "Task",   "InstanceRole": "TASK",
             "InstanceType": "m5.2xlarge", "InstanceCount": 8,
             "Market": "SPOT", "BidPrice": "0.25"},
        ],
        "Ec2SubnetId": SUBNET_ID,
        "KeepJobFlowAliveWhenNoSteps": True,
        "TerminationProtected": False,
    },
    "Configurations": [
        {"Classification": "spark-defaults", "Properties": {
            "spark.sql.shuffle.partitions":           "200",
            "spark.dynamicAllocation.enabled":        "true",
            "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
            "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        }},
    ],
    "JobFlowRole":       "EMR_EC2_DefaultRole",
    "ServiceRole":       "EMR_DefaultRole",
    "Tags":              [{"Key": "env", "Value": ENV}, {"Key": "pipeline", "Value": "shopstream"}],
    "VisibleToAllUsers": True,
}


def make_step(name: str, script: str, run_date: str) -> dict:
    return {
        "Name": name,
        "ActionOnFailure": "CONTINUE",
        "HadoopJarStep": {
            "Jar": "command-runner.jar",
            "Args": [
                "spark-submit", "--deploy-mode", "cluster",
                "--py-files",   CODE_URI,
                f"s3://{CODE_BUCKET}/{GIT_SHA}/{script}",
                "--run_date",   run_date,
                "--env",        ENV,
                "--data_bucket", DATA_BUCKET,
            ],
        },
    }


def check_dq_results(run_date: str, **ctx):
    import boto3, json
    s3  = boto3.client("s3")
    key = f"dq_metrics/silver_{run_date}.json"
    try:
        obj = s3.get_object(Bucket=DATA_BUCKET, Key=key)
        dq  = json.loads(obj["Body"].read())
    except s3.exceptions.NoSuchKey:
        raise ValueError(f"DQ metrics not found: s3://{DATA_BUCKET}/{key}")
    if not dq.get("passed"):
        raise ValueError(f"Silver DQ failed: {dq.get('violations')}")
    print(f"Silver DQ passed: {dq}")


# ── DAG definition ────────────────────────────────────────────────────────
default_args = {
    "owner":            "data-eng",
    "retries":          1,
    "retry_delay":      timedelta(minutes=5),
    "email_on_failure": True,
    "email":            ["data-eng-alerts@shopstream.com"],
    "sla":              timedelta(hours=2),
}

with DAG(
    dag_id="shopstream_pipeline",
    default_args=default_args,
    schedule_interval="0 6 * * *",      # daily at 6am UTC
    start_date=datetime(2024, 1, 1),
    catchup=False,
    max_active_runs=1,
    tags=["shopstream", "production", "medallion"],
    doc_md="""### ShopStream Daily Batch Pipeline\n
    Bronze validate → Silver transform → Gold aggregate → freshness check.
    Runs on EMR with Spot task nodes. Terminates cluster after all steps (success or fail).""",
) as dag:

    run_date = "{{ ds }}"   # Airflow logical date (YYYY-MM-DD)

    # ── 0. Wait for Bronze data ────────────────────────────────
    wait_bronze = S3KeySensor(
        task_id        = "wait_for_bronze",
        bucket_name    = DATA_BUCKET,
        bucket_key     = f"bronze/orders/date={{{{ ds }}}}/",
        wildcard_match = True,
        poke_interval  = 900,   # 15 min
        timeout        = 7200,  # 2 hours max wait
        mode           = "reschedule",  # release worker slot between pokes
    )

    # ── 1. Create EMR cluster ─────────────────────────────────
    create_cluster = EmrCreateJobFlowOperator(
        task_id         = "create_emr_cluster",
        job_flow_overrides = EMR_CONFIG,
        aws_conn_id     = "aws_default",
    )

    # ── 2. Add Spark steps ────────────────────────────────────
    with TaskGroup(group_id="emr_steps") as emr_steps:
        scripts = [
            ("validate_bronze",  "validate_bronze.py"),
            ("silver_transform", "silver_transform.py"),
            ("gold_aggregate",   "gold_aggregate.py"),
            ("freshness_check",  "freshness_check.py"),
        ]
        prev_sensor = None
        for step_name, script in scripts:
            add = EmrAddStepsOperator(
                task_id      = f"add_{step_name}",
                job_flow_id  = "{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
                steps        = [make_step(step_name, script, run_date)],
                aws_conn_id  = "aws_default",
            )
            wait = EmrStepSensor(
                task_id     = f"wait_{step_name}",
                job_flow_id = "{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
                step_id     = f"{{{{ task_instance.xcom_pull('emr_steps.add_{step_name}', key='return_value')[0] }}}}",
                aws_conn_id = "aws_default",
                mode        = "reschedule",
                poke_interval = 60,
            )
            add >> wait
            if prev_sensor:
                prev_sensor >> add
            prev_sensor = wait

    # ── 3. Terminate EMR (always — even on failure) ───────────
    terminate_cluster = EmrTerminateJobFlowOperator(
        task_id      = "terminate_emr_cluster",
        job_flow_id  = "{{ task_instance.xcom_pull('create_emr_cluster', key='return_value') }}",
        aws_conn_id  = "aws_default",
        trigger_rule = TriggerRule.ALL_DONE,  # run even if steps fail
    )

    # ── 4. Airflow-side DQ check ──────────────────────────────
    dq_check = PythonOperator(
        task_id         = "check_silver_dq",
        python_callable = check_dq_results,
        op_kwargs       = {"run_date": run_date},
    )

    # ── Task dependencies ─────────────────────────────────────
    wait_bronze >> create_cluster >> emr_steps >> terminate_cluster >> dq_check
""")

## Part 2: Production Spark Job Template

Each EMR Step runs one Spark script. Every script follows the same structure: parse args → load config → run → DQ → write metrics → exit.

In [ ]:
print("""
# silver_transform.py — deployed to s3://shopstream-code/{git-sha}/
# ══════════════════════════════════════════════════════════════════
import argparse, json, logging, sys, time
from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from delta.tables import DeltaTable

# ── Structured logging ────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("shopstream")

def info(msg, run_id, **kw):
    log.info(json.dumps({
        "ts": datetime.now(timezone.utc).isoformat(),
        "level": "INFO", "pipeline": "shopstream",
        "layer": "silver", "run_id": run_id, "msg": msg, **kw
    }))

def error(msg, run_id, **kw):
    log.error(json.dumps({
        "ts": datetime.now(timezone.utc).isoformat(),
        "level": "ERROR", "pipeline": "shopstream",
        "layer": "silver", "run_id": run_id, "msg": msg, **kw
    }))


# ── Config ────────────────────────────────────────────────────
CONFIGS = {
    "dev":  {"silver_path": "s3://shopstream-data-dev/silver/orders",
             "dead_letter": "s3://shopstream-data-dev/dead_letter/orders",
             "dq_path":     "s3://shopstream-data-dev/dq_metrics"},
    "prod": {"silver_path": "s3://shopstream-data/silver/orders",
             "dead_letter": "s3://shopstream-data/dead_letter/orders",
             "dq_path":     "s3://shopstream-data/dq_metrics"},
}

VALID_STATUSES = ["shipped", "pending", "cancelled"]


# ── Pure transforms (tested in pytest) ───────────────────────
def split_valid_dead_letter(df):
    reason = (
        F.when(F.col("user_id").isNull(), F.lit("|null_user_id")).otherwise(F.lit("")) +
        F.when(F.col("amount") <= 0,      F.lit("|non_positive_amount")).otherwise(F.lit("")) +
        F.when(~F.col("status").isin(VALID_STATUSES), F.lit("|invalid_status")).otherwise(F.lit(""))
    )
    tagged      = df.withColumn("_rejection_reason", reason)
    valid       = tagged.filter(F.col("_rejection_reason") == "").drop("_rejection_reason")
    dead_letter = tagged.filter(F.col("_rejection_reason") != "") \
                        .withColumn("_rejected_at", F.current_timestamp())
    return valid, dead_letter

def deduplicate(df):
    from pyspark.sql import Window
    w = Window.partitionBy("order_id").orderBy("event_time")
    return df.withColumn("_rn", F.row_number().over(w)) \
             .filter(F.col("_rn") == 1).drop("_rn")

def enrich(df):
    return df \
        .withColumn("event_ts",  F.to_timestamp("event_time")) \
        .withColumn("amount",    F.col("amount").cast(DoubleType())) \
        .withColumn("tier",
            F.when(F.col("amount") >= 1000, "premium")
             .when(F.col("amount") >= 300,  "standard")
             .otherwise("basic")) \
        .select("order_id","user_id","product","amount","status","region","tier","event_ts","date")


# ── Main ──────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--run_date",    required=True)
    parser.add_argument("--env",         default="dev")
    parser.add_argument("--data_bucket", required=True)
    args   = parser.parse_args()
    cfg    = CONFIGS[args.env]
    run_id = f"run_{args.run_date.replace('-', '')}_001"

    spark = SparkSession.builder \
        .appName(f"ShopStream Silver — {args.run_date}") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()

    t0 = time.time()
    info("silver_start", run_id=run_id, run_date=args.run_date)

    try:
        bronze = spark.read.format("delta") \
            .load(f"s3://{args.data_bucket}/bronze/orders") \
            .filter(F.col("date") == args.run_date)

        valid, dead = split_valid_dead_letter(bronze)
        silver = valid.pipe(deduplicate).pipe(enrich)

        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
        silver.write.format("delta").mode("overwrite") \
              .partitionBy("date").save(cfg["silver_path"])
        dead.write.format("delta").mode("append") \
            .partitionBy("date").save(cfg["dead_letter"])

        n_valid = silver.count()
        n_dead  = dead.count()
        dq = {"passed": True, "run_date": args.run_date,
              "valid_rows": n_valid, "rejected_rows": n_dead,
              "violations": []}

        # Write DQ metrics for Airflow to check
        import boto3
        s3 = boto3.client("s3")
        s3.put_object(
            Bucket=args.data_bucket,
            Key=f"dq_metrics/silver_{args.run_date}.json",
            Body=json.dumps(dq),
        )

        info("silver_complete", run_id=run_id,
             valid_rows=n_valid, rejected_rows=n_dead,
             duration_s=round(time.time()-t0, 2))

    except Exception as e:
        error("silver_failed", run_id=run_id, error=str(e))
        sys.exit(1)   # non-zero exit → EMR Step FAILED → Airflow detects


if __name__ == "__main__":
    main()
""")

## Part 3: GitHub Actions CI/CD Workflow

The full CI/CD pipeline: lint → test → deploy on merge.

In [ ]:
print("""
# .github/workflows/ci.yml
# ══════════════════════════════════════════════════════════════
name: ShopStream CI/CD

on:
  pull_request:
    branches: [main]
  push:
    branches: [main]

env:
  PYTHON_VERSION: '3.11'
  JAVA_VERSION:   '17'

jobs:

  # ── Job 1: Lint ──────────────────────────────────────────────
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - name: Install ruff
        run: pip install ruff

      - name: Lint
        run: ruff check src/ tests/

  # ── Job 2: Unit Tests ─────────────────────────────────────────
  test:
    runs-on: ubuntu-latest
    needs: lint
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: ${{ env.PYTHON_VERSION }}
          cache: pip

      - uses: actions/setup-java@v4
        with:
          java-version: ${{ env.JAVA_VERSION }}
          distribution: temurin
          cache: maven

      - name: Install dependencies
        run: pip install -r requirements-dev.txt

      - name: Run unit tests
        run: pytest tests/unit/ -v --cov=src --cov-report=xml --cov-fail-under=80

      - name: Upload coverage
        uses: codecov/codecov-action@v4
        with:
          files: coverage.xml

  # ── Job 3: Deploy (merge to main only) ───────────────────────
  deploy:
    runs-on: ubuntu-latest
    needs: test
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    permissions:
      id-token: write   # required for OIDC
      contents: read

    steps:
      - uses: actions/checkout@v4

      - name: Configure AWS via OIDC (no stored keys!)
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: arn:aws:iam::123456789012:role/shopstream-github-deploy
          aws-region:     us-east-1

      - name: Package and deploy code to S3
        run: |
          GIT_SHA=$(git rev-parse --short HEAD)
          echo "Deploying SHA: $GIT_SHA"

          # Package PySpark src as zip (importable by --py-files)
          cd src && zip -r ../libs.zip . && cd ..

          # Upload code
          aws s3 cp libs.zip          s3://shopstream-code/$GIT_SHA/libs.zip
          aws s3 cp src/silver_transform.py s3://shopstream-code/$GIT_SHA/silver_transform.py
          aws s3 cp src/gold_aggregate.py   s3://shopstream-code/$GIT_SHA/gold_aggregate.py
          aws s3 cp dags/shopstream_pipeline.py s3://shopstream-code/dags/shopstream_pipeline.py

          # Update Airflow Variable so next DAG run uses this SHA
          aws ssm put-parameter \\
            --name "/shopstream/prod/GIT_SHA" \\
            --value "$GIT_SHA" \\
            --type String --overwrite

          echo "Deployed $GIT_SHA to s3://shopstream-code/$GIT_SHA/"
""")

## Part 4: Unit Tests for the Pipeline

These tests run in CI and verify the three pure transform functions.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Use existing session
spark = SparkSession.builder \
    .appName("ShopStream Capstone - Orchestration") \
    .master("local[2]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

VALID_STATUSES = ["shipped", "pending", "cancelled"]

def split_valid_dead_letter(df):
    reason = (
        F.when(F.col("user_id").isNull(), F.lit("|null_user_id")).otherwise(F.lit("")) +
        F.when(F.col("amount") <= 0,      F.lit("|non_positive_amount")).otherwise(F.lit("")) +
        F.when(~F.col("status").isin(VALID_STATUSES), F.lit("|invalid_status")).otherwise(F.lit(""))
    )
    tagged      = df.withColumn("_rejection_reason", reason)
    valid       = tagged.filter(F.col("_rejection_reason") == "").drop("_rejection_reason")
    dead_letter = tagged.filter(F.col("_rejection_reason") != "") \
                        .withColumn("_rejected_at", F.current_timestamp())
    return valid, dead_letter

def deduplicate(df):
    from pyspark.sql import Window
    w = Window.partitionBy("order_id").orderBy("event_time")
    return df.withColumn("_rn", F.row_number().over(w)) \
             .filter(F.col("_rn") == 1).drop("_rn")

def enrich(df):
    return df \
        .withColumn("event_ts", F.to_timestamp("event_time")) \
        .withColumn("amount",   F.col("amount").cast(DoubleType())) \
        .withColumn("tier",
            F.when(F.col("amount") >= 1000, "premium")
             .when(F.col("amount") >= 300,  "standard")
             .otherwise("basic")) \
        .select("order_id","user_id","product","amount","status","region","tier","event_ts","date")


# ── Inline test suite (mirrors tests/unit/test_silver.py) ─────
COLS = ["order_id", "user_id", "product", "amount", "status", "region", "event_time", "date"]
passed = 0
failed = 0

def test(name, cond):
    global passed, failed
    if cond:
        print(f"  PASS  {name}")
        passed += 1
    else:
        print(f"  FAIL  {name}")
        failed += 1

print("=" * 55)
print("Running ShopStream unit tests")
print("=" * 55)

# ── split_valid_dead_letter tests ─────────────────────────────
print("\n[split_valid_dead_letter]")

good = spark.createDataFrame([
    ("O1", "U1", "laptop",  999.0, "shipped",  "US", "2024-01-15 09:00:00", "2024-01-15"),
    ("O2", "U2", "phone",   200.0, "pending",  "EU", "2024-01-15 09:01:00", "2024-01-15"),
], COLS)
v, d = split_valid_dead_letter(good)
test("all valid → 2 valid, 0 dead", v.count() == 2 and d.count() == 0)

bad = spark.createDataFrame([
    ("O3", None, "kb",  10.0, "shipped",  "US", "2024-01-15 09:02:00", "2024-01-15"),  # null user
    ("O4", "U4", "m",  -1.0, "shipped",  "US", "2024-01-15 09:03:00", "2024-01-15"),  # negative
    ("O5", "U5", "c",   5.0, "UNKNOWN",  "US", "2024-01-15 09:04:00", "2024-01-15"),  # bad status
], COLS)
v2, d2 = split_valid_dead_letter(bad)
test("all bad → 0 valid, 3 dead", v2.count() == 0 and d2.count() == 3)
test("dead_letter has _rejection_reason", "_rejection_reason" in d2.columns)
test("dead_letter has _rejected_at",      "_rejected_at" in d2.columns)

empty = spark.createDataFrame([], good.schema)
v3, d3 = split_valid_dead_letter(empty)
test("empty df → 0 valid, 0 dead", v3.count() == 0 and d3.count() == 0)

# ── deduplicate tests ─────────────────────────────────────────
print("\n[deduplicate]")

dupes = spark.createDataFrame([
    ("O1", "U1", "laptop", 999.0, "shipped", "US", "2024-01-15 09:00:00", "2024-01-15"),
    ("O1", "U1", "laptop", 999.0, "shipped", "US", "2024-01-15 09:00:00", "2024-01-15"),
    ("O2", "U2", "phone",  200.0, "pending", "EU", "2024-01-15 09:01:00", "2024-01-15"),
], COLS)
deduped = deduplicate(dupes)
test("dedup removes duplicate: 3 → 2", deduped.count() == 2)
test("dedup: O1 appears once", deduped.filter(F.col("order_id") == "O1").count() == 1)

# ── enrich tests ──────────────────────────────────────────────
print("\n[enrich]")

to_enrich = spark.createDataFrame([
    ("O1", "U1", "laptop",   1299.0, "shipped",   "US", "2024-01-15 09:00:00", "2024-01-15"),
    ("O2", "U2", "monitor",   329.0, "shipped",   "EU", "2024-01-15 09:05:00", "2024-01-15"),
    ("O3", "U3", "headphones", 99.0, "cancelled", "US", "2024-01-15 09:10:00", "2024-01-15"),
], COLS)
enriched = enrich(to_enrich)
rows = {r["order_id"]: r for r in enriched.collect()}
test("O1 (1299) → premium",  rows["O1"]["tier"] == "premium")
test("O2 (329)  → standard", rows["O2"]["tier"] == "standard")
test("O3 (99)   → basic",    rows["O3"]["tier"] == "basic")
test("event_ts is timestamp", str(enriched.schema["event_ts"].dataType) == "TimestampType()")
test("amount is double",      str(enriched.schema["amount"].dataType) == "DoubleType()")

print()
print("=" * 55)
print(f"Results: {passed} passed, {failed} failed")
print("=" * 55)
if failed > 0:
    raise AssertionError(f"{failed} test(s) failed")

## Exercises

1. The Airflow DAG currently has `retries=1` and `retry_delay=5m`. The Silver transform step fails 30% of the time on the first run due to transient S3 throttling. What values would you set for `retries` and `retry_delay`, and why? Also add exponential backoff.
2. Add a `BranchPythonOperator` after `wait_for_bronze` that checks if the Bronze partition exists AND has at least 1,000 rows. If not, branch to a `send_alert` task (skip processing); if yes, proceed to `create_emr_cluster`.
3. The deploy workflow currently overwrites the DAG file in S3 (`shopstream_pipeline.py`) on every merge. Propose a safer blue/green deployment strategy where the new DAG is staged and only promoted to MWAA after a manual approval step. Sketch the GitHub Actions changes.
4. Write a `test_gold_aggregate.py` (pytest) that tests a `build_gold_revenue_by_region()` function: (a) correct sum for multiple regions, (b) empty Silver → empty Gold, (c) single region → one row. Use a session-scoped SparkSession fixture.
5. The `sys.exit(1)` in `silver_transform.py` causes EMR Step to show as FAILED. How does Airflow detect this? Trace the chain: EMR Step status → `EmrStepSensor` → Airflow task state → DAG run state → notification.